# Laboration 1: Intelligenta agenter


Begin by running the two example agents `RandomAgent` and `ReflexAgentWithState`. Try to understand what the different parts of the code does. Stop the agent using the `Stop` button above or by closing the Pacman window.

The actions that can be performed by a Pacman agent are: 
- *GoForward*: Take one step forward
- *GoRight*: Turn 90 degrees right and take one step forward
- *GoLeft*: Turn 90 degrees left and take one step forward
- *GoBack*: Turn and 180 degrees and take one step forward
- *Stop*: Shut down the agent
- Any other command: No effect

*Note: If the the Pacman window freezes, you can still (usually) re-run the code. If this does not work, try `Kernel/Restart`, or restart the Jupyter Notebook* 

In [1]:
class RandomAgent(BaseAgent):

    class State:
        def __init__(self):
            self.actions = ["GoRight", "GoLeft", "GoForward", "GoBack"]

    def choose_action(self, state):
        action = random.choice(state.actions)
        print("Performing action:", action)
        return action

NameError: name 'BaseAgent' is not defined

In [ ]:
# Run RandomAgent in the room layout "layouts/custom.lay"
args = readCommand(["--pacman", RandomAgent,
                    "--layout", "mediumEmpty"])
runGames(**args)

In [ ]:
class ReflexAgentWithState(BaseAgent):
    class State:
        def __init__(self):
            self.bump = False
            self.previous_action = ""
            self.actions = ["GoRight", "GoLeft", "GoForward", "GoBack"]

        def __repr__(self):
            if self.bump:
                return self.previous_action + " resulted in a bump"
            else:
                return self.previous_action
    
    def update_state_with_percept(self, percept, state):
        if percept[1] == "bump":
            state.bump = True
        else:
            state.bump = False
        return state

    def choose_action(self, state):
        actions = state.actions
        if state.bump:
            actions.remove(state.previous_action)
        return random.choice(actions)

    def update_state_with_action(self, action, state):
        state.previous_action = action
        # Print the representation (i.e. __repr__) of the state
        print(state)
        return state

In [ ]:
# Run ReflexAgentWithState
args = readCommand(["--pacman", ReflexAgentWithState,
                    "--layout", "mediumEmpty"])
runGames(**args)

**Exercise 1**: Briefly describe the behavior of the two agents above. What is the main difference between the two?

**Answer:**   : RandomAgent just goes in a random direction everytime, while ReflexAgentWithState, if the previous   action resulted in a collision with the wall, doesn't go in that direction again immediately after.

**Exercise 2**: Devise a strategy that will allow the agent to eat all the food in the room and then stop. Discuss the strategy with your partner. Remember to test your strategy in 'odd' worlds, such as worlds of the size 1x1 or 1x5. Describe your strategy in general terms below.

**Answer:**   : The agent spawns in the bottom-left corner and starts moving upwards. As soon as it hits the wall, it turns right once and then starts going downwards until it hits the wall again and then turns left once and then starts going upwards again, repeating the process. This continues until the agent bumps into the wall twice in a row because this means it has hit a corner after clearing the right-most wall. This way, the agent will clear any map regardless of how it is structured.

In [2]:
from agents import *
from pacman import *

In [132]:
class AgentWithState(BaseAgent):

    def __init__(self):
        pass

        
    class State:
        def __init__(self, latest_turn="Right", 
                     map_length=-1, map_height=0, 
                     step_counter=0, step_counter_max=0, 
                     line_counter=0, line_counter_max=0, 
                     initial_left_turn=None, is_turning=False, 
                     phase="length_scan", first_state=True,
                     stop_search=False):

            # The agent must remember its latest turning action 
            # as the next one will be in the opposite direction.
            self.latest_turn = latest_turn
            
            # The agent will remember the size of the map so it 
            # knows when to turn.
            self.map_length = map_length
            self.map_height = map_height
            
            # The step_counter and step_counter_max determines the agent's
            # turning condition during the clearing phase.
            self.step_counter = step_counter
            self.step_counter_max = step_counter_max
            
            # The line_counter and line_counter_max determine the agent's
            # stopping condition if it ever comes to the clearing phase.
            self.line_counter = line_counter
            self.line_counter_max = line_counter_max
            
            # If the agent chooses to clear the map vertically it has to
            # turn left at the initiation of the clearing phase, whether
            # the agent turns or not is determined by this variable:
            self.initial_left_turn = initial_left_turn
            
            # When the agent turns during the clearing phase, it must turn
            # twice so it has to keep in its memory whether the last action
            # was a turn.
            self.is_turning = is_turning
            
            # The clearing consist of three phases: "length_scan", "height_scan" and "clear" in 
            # which the agent will behave differently.
            self.phase = phase

            self.bump = False
            self.previous_action = ""
            self.stop_search = stop_search

        def __repr__(self):
            if self.bump:
                return self.previous_action + " resulted in a bump"
            else:
                return self.previous_action

            
    def update_state_with_percept(self, percept, state):
        """Update the state based on percept"""
        
        if percept[1] == "bump":
            state.bump = True
        else:
            state.bump = False
        

        # The phase where the agent scans the length of the map.
        if state.phase == "length_scan":
            state.map_length += 1
            
            # When the agent reaches the wall, it will turn left and scan upwards instead.
            if state.bump:
                state.phase = "height_scan"


        # The phase where the agent scans the height of the map.
        elif state.phase == "height_scan":
            
            state.map_height += 1
            
            # When the agent reaches the wall, the size of the map
            # is known and the clearing phase can start.
            if state.bump:

                state.phase = "clear"
                
                # The agent's turning condition will be determined by the
                # map's height if the map is more high than wide.
                state.step_counter_max = state.map_length - 2
                    
                # The agent's turning condition will be determined by the
                # map's length if the map is more high than wide.
                state.line_counter_max = state.map_height - 2
                    
                # The agent's first turn will be to the left if it is to
                # clear the map vertically, after it has turned left on
                # the start of the clearing phase.
                state.latest_turn = "Right"
                state.initial_left_turn = True


        # The phase where the agent with the knowledge of the map size, clears the remains of it.
        elif state.phase == "clear":
            
            # The agent always performs turning actions twice, therefore we
            # have to check if it is in the middle of a turn.
            if state.is_turning:
                
                # Set this to false so the agent doesn't turn more than twice.
                state.is_turning = False
                
                # With the knowledge of the direction the last turn was in,
                # determine which direction the next turn is going to be.
                # Also remember which turn it just made so the next turn can
                # be in the opposite direction.
                if state.latest_turn == "Right":
                    state.latest_turn = "Left"
                elif state.latest_turn == "Left":
                    state.latest_turn = "Right"
            
            # If the step counter has reached it's maximal value,
            # the agent's turning condition has been fulfilled.
            if state.step_counter == state.step_counter_max:
                
                # If the agent is about to turn, it means that an
                # entire line has been surpassed.
                state.line_counter += 1
                
                # Reset the step counter so the agent doesn't get
                # stuck in the turning condition.
                state.step_counter = 0
                
                # Make the agent remember that it has to turn again next state.
                state.is_turning = True
            else:
                # If the agent is about to go in a direction which isn't tangentical
                # to the clearing direction, add one to the step counter.
                if not state.initial_left_turn:
                    state.step_counter += 1
            
            
            # If the map is to be cleared vertically, the agent has to do an 
            # inital turn to the left before starting to go forward.
            if state.initial_left_turn:
                state.initial_left_turn = False

        return state


    def choose_action(self, state):
        """Return an action"""

        if state.stop_search:
            return "Stop"

        # The phase where the agent scans the length of the map.
        if state.phase == "length_scan":
            
            # When the agent reaches the wall, it will turn left and scan upwards instead.
            if state.bump:
                return "GoLeft"
            
            return "GoForward"


        # The phase where the agent scans the height of the map.
        if state.phase == "height_scan":
            
            # When the agent reaches the wall, the size of the map
            # is known and the clearing phase can start.
            if state.bump:
                
                # If the map has the height or length of one, the clearing phase 
                # doesn't have to be initiated since the entire map has already been covered.
                #if state.map_length == 1 or state.map_height == 1:
                    #return "Stop"
                
                # Do a left turn before the clearing phase to avoid another bump.
                return "GoLeft"
            
            # If the agent hasn't yet reached the wall, just keep going forward.
            return "GoForward"


        # The phase where the agent with the knowledge of the map size, clears the remains of it.
        if state.phase == "clear":

            # High map edge case
            if state.map_length == 1 or state.map_height == 1:
                return "Stop"

            # This can only be the case on a 2x2 map, and at this point it
            # means that the agent has cleared the entire map.
            if state.step_counter_max == 0:
                return "Stop"

            # If the step counter has reached it's maximal value,
            # the agent's turning condition has been fulfilled.
            if state.step_counter == state.step_counter_max:
                # This means that the end of the map has been reached and
                # the agent therefore stops to avoid an unecessary bump.
                if state.line_counter == state.line_counter_max:
                    return "Stop"
                
                # With the knowledge of the direction the last turn was in,
                # determine which direction the next turn is going to be.
                if state.latest_turn == "Right":
                    return "GoLeft"
                elif state.latest_turn == "Left":
                    return "GoRight"
            
            # The agent always performs turning actions twice, therefore we
            # have to check if it is in the middle of a turn.
            if state.is_turning:
                
                # With the knowledge of the direction the last turn was in,
                # determine which direction the next turn is going to be.
                # Also remember which turn it just made so the next turn can
                # be in the opposite direction.
                if state.latest_turn == "Right":
                    return "GoLeft"
                elif state.latest_turn == "Left":
                    return "GoRight"
            
            # If the map is to be cleared vertically, the agent has to do an 
            # inital turn to the left before starting to go forward.
            if state.initial_left_turn:
                return "GoLeft"
            
            # If no other condition is satisfied, the agent just keeps on moving forward.
            return "GoForward"


    def update_state_with_action(self, action, state):
        state.previous_action = action
        # Print the representation (i.e. __repr__) of the state
        print(state)
        return state

In [133]:
# Run AgentWithState. Try different layouts in the layout directory.
#Consider adding your own layouts to the ./layout directory to debug your code.
args = readCommand(["--pacman", AgentWithState,
                    "--layout", "largeEmpty"])
runGames(**args)

GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoLeft resulted in a bump
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoLeft resulted in a bump
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoLeft
GoLeft
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoRight
GoRight
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForward
GoForwar

**Exercise 3**: Implement the strategy you described in the previous exercise by extending `AgentWithState` above. Remember to comment and format the code appropriately.

**VG only:** Make sure your strategy prevents your agent from bumping more times than strictly necessary (beware of layouts 1x10, 10x1, 1x1).

**Exercise 4:** What environments are the agent best suited for? Describe the environment using terminology from the course literature (see Russel and Norvig, ch. 2).

**Answer:** The agent was made with a partially-observable, deterministic, sequential, static and discrete environment in consideration, therefore it performs best in these environmental conditions. The agent cannot observe the environment fully; only the coordinate which the agent is on can be observed. Although, based off our implementation of the clearing strategy, the initial actions the agent performs could be seen as a sort of psuedo-observation of the environment, given its restrictions regarding observation. If the agent could observe the environment fully it would be possible to devise a more optimal strategy to complete the course. Also the environment is deterministic because the agents actions are determined by it's previous actions. If the environment were not deterministic the current agent likely would not be able to complete the course because the rules upon which the agent acts are designed to only accomodate deterministic environments. The environment also needs to be static, meaning the agent will not perform optimally in an environmet which changes during the activity. Given that the agent is computer simulated it is given that the environment is discrete, meaning it is not plausible for the agent to act in a continuous environment.

**Exercise 5**: Compare your agent to the predefined ones. How do they differ? Is your agent more intelligent than the other two? Is it more rational? Motivate your answers.

**Answer:** The implementation on exercise 2 can be seen as more rational than the previous two agents because the strategy it implements is more optimal. It also incorporates a basic observation of the environment, although only the dimensions of the map, which contributes to which execution strategy the agent chooses to clear the map. On these conditions the agent can be seen as more rational than the other two. Its intelligence, can however not be seen as significantly greater. The agent chooses what strategy to use while clearing the map, this decision process can be seen as a form of intelligence. However if this decision process constitutes intelligence can be doubted. Our implementation differs from the other agents because it has rules and conditional rules that the other agents do not. The second agent does have conditional rules, but our agent differs on the point that its conditional rules are greatly expanded.

**Exercise 6 (VG):** Discuss whether your agent's strategy is optimal or not. Could it be improved in any way given the current restrictions?

**Answer:** Based on our testing of the strategy we believe our implementation is optimal in ensuring the minimal ammount of bumps and moves. Because the agent cannot observe the environment our strategy ensures a minimal ammount of bumps. However if the map would be x=1, y=10 in size there would be one initial bump that would not occur if the x and y coordinates were swapped. Because the agent cannot percieve the map it cannot choose the more optimal choice of initially going upwards in this scenario. The choice to have the agent act this way is based on the fact that the agent spawns facing east, therefore our implementation ensures a minimal ammount of moves in every other kind of case. If it were implemented in the strategy that the agent would initially go upwards the same problem would arise if the map was more wide than high. 

Given the arguments presented, we believe our agent is optimal given the current restrictions.